<a href="https://colab.research.google.com/github/Imran1hp/BPE-Tokenizer_transformer/blob/main/Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Download the dataset

In [1]:
import torch

In [2]:
!pip install datasets --quiet

In [3]:
from datasets import load_dataset
data_set = load_dataset("Salesforce/wikitext",
    "wikitext-2-raw-v1")


In [4]:
print(data_set)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [5]:
train_text = "\n".join(data_set["train"]["text"])
print(f"Length of the train set {len(train_text)}")



Length of the train set 10929707


#Tokenizing the dataset

In [6]:
tokens= train_text.encode("utf-8")
tokens = list(map(int , tokens))

Checking how many time a pair is repeating

In [7]:
def get_stats(ids):
  counts = {}
  for pair in zip(ids , ids[1:]):
   counts[pair]= counts.get(pair ,0)+1

  return counts

Merging the frequently occuring pair

In [8]:

def merge(ids , pair , new_id):

  new_ids =[]
  i =0
  while i <len(ids):
    if i < len(ids)-1 and ids[i] == pair[0] and ids[i+1] ==pair[1]:
      new_ids.append(new_id)
      i+=2
    else:
      new_ids.append(ids[i])
      i+=1

  return new_ids




#Itering to merge the frequent pairs

In [9]:

def token_merger(VOCAB_SIZE:int, tokens:list):

  INITIAL_VOCAB_SIZE = 256
  NUMBER_MERGES = VOCAB_SIZE - INITIAL_VOCAB_SIZE

  merges= {}

  for i in range(NUMBER_MERGES):
    stats = get_stats(tokens)
    pair = max(stats , key = stats.get)

    new_id = INITIAL_VOCAB_SIZE + i
    print(f"Merging {pair} with {new_id}")
    tokens = merge(tokens , pair ,new_id)
    merges[pair]= new_id

  return tokens , merges






In [10]:
VOCAB_SIZE = 1000
tokens = tokens[:1000000]
print(f" before Length of the token {len(tokens)}")
tokens , merges = token_merger(VOCAB_SIZE ,tokens)
print(f"After lenght of the tokens {len(tokens)}")

 before Length of the token 1000000
Merging (101, 32) with 256
Merging (115, 32) with 257
Merging (116, 104) with 258
Merging (100, 32) with 259
Merging (110, 32) with 260
Merging (101, 114) with 261
Merging (116, 32) with 262
Merging (258, 256) with 263
Merging (105, 110) with 264
Merging (44, 32) with 265
Merging (97, 110) with 266
Merging (101, 259) with 267
Merging (121, 32) with 268
Merging (111, 114) with 269
Merging (97, 114) with 270
Merging (46, 32) with 271
Merging (97, 108) with 272
Merging (116, 105) with 273
Merging (111, 32) with 274
Merging (114, 101) with 275
Merging (102, 32) with 276
Merging (111, 110) with 277
Merging (261, 32) with 278
Merging (111, 276) with 279
Merging (103, 32) with 280
Merging (97, 32) with 281
Merging (101, 110) with 282
Merging (266, 259) with 283
Merging (111, 260) with 284
Merging (264, 280) with 285
Merging (105, 260) with 286
Merging (97, 257) with 287
Merging (111, 117) with 288
Merging (115, 116) with 289
Merging (116, 274) with 290
Merg

In [11]:
len(tokens)
compression =  1000000 /int(len(tokens))
compression

2.6563176530902273

#Split the data

In [12]:
train_end = int(len(tokens) * .90)
val_end = int(len(tokens) *0.95)
train_ids = tokens[:train_end]
val_ids = tokens[train_end:val_end]
test_ids = tokens[val_end:]
print(f"Length of train_ids {len(train_ids)}")
print(f"Length of val_ids {len(val_ids)}")
print(f"Length of test_ids {len(test_ids)}")

Length of train_ids 338814
Length of val_ids 18823
Length of test_ids 18824


#Decoding the tokenizer

In [13]:
def build_vocad(merges):

  byte_vocab = {idx :bytes([idx]) for idx in range(256)}

  for (p0 ,p1), idx in merges.items():
    byte_vocab[idx] = byte_vocab[p0] + byte_vocab[p1]

  return byte_vocab



In [21]:
vocab = build_vocad(merges)
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [15]:
def decoder(ids, vocab = vocab):

  tokens = b"".join(vocab[idx] for idx in ids)
  text = tokens.decode("utf-8", errors='replace')

  return text

In [16]:
decoded_text = decoder([776 ,32, 788 ,32 ,32, 555 , 32, 873,  32, 999], vocab)

print(decoded_text)

produc high  ost  ship posi


In [17]:
def encoder(text):
  tokens = list(text.encode("utf-8"))
  while len(tokens) >=2:
    stats = get_stats(tokens)
    pair = min(stats ,key = lambda p :merges.get(p , float("inf")))
    if pair not in merges:
      break;

    idx = merges[pair]

    tokens = merge(tokens , pair , idx)
  return tokens

In [18]:
list("h".encode("utf-8"))

[104]

In [19]:
encoder("h")

[104]

In [23]:
print(decoder(encoder("Hello world.i am imran")))

Hello world.i am imran
